In [1]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community


Using Python 3.12.1 environment at: E:\1.3.5.7.9\2.LLMops\.venv
Checked 6 packages in 653ms


In [2]:
import os
from google.genai import Client

# Get API key from environment variable
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Please set the GOOGLE_API_KEY environment variable.")

# Initialize client with the API key from env
client = Client(api_key=api_key)

# List available models
models = client.models.list()

for m in models:
    print("Name:", m.name)
    print("Display Name:", m.display_name)
    # Capabilities might contain supported generation methods
    print("Capabilities:", getattr(m, "capabilities", "N/A"))
    print("---")

Name: models/gemini-2.5-flash
Display Name: Gemini 2.5 Flash
Capabilities: N/A
---
Name: models/gemini-2.5-pro
Display Name: Gemini 2.5 Pro
Capabilities: N/A
---
Name: models/gemini-2.0-flash
Display Name: Gemini 2.0 Flash
Capabilities: N/A
---
Name: models/gemini-2.0-flash-001
Display Name: Gemini 2.0 Flash 001
Capabilities: N/A
---
Name: models/gemini-2.0-flash-lite-001
Display Name: Gemini 2.0 Flash-Lite 001
Capabilities: N/A
---
Name: models/gemini-2.0-flash-lite
Display Name: Gemini 2.0 Flash-Lite
Capabilities: N/A
---
Name: models/gemini-2.5-flash-preview-tts
Display Name: Gemini 2.5 Flash Preview TTS
Capabilities: N/A
---
Name: models/gemini-2.5-pro-preview-tts
Display Name: Gemini 2.5 Pro Preview TTS
Capabilities: N/A
---
Name: models/gemma-3-1b-it
Display Name: Gemma 3 1B
Capabilities: N/A
---
Name: models/gemma-3-4b-it
Display Name: Gemma 3 4B
Capabilities: N/A
---
Name: models/gemma-3-12b-it
Display Name: Gemma 3 12B
Capabilities: N/A
---
Name: models/gemma-3-27b-it
Display 

In [3]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from .env
load_dotenv()

# Initialize the LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",                   # correct parameter
    google_api_key=os.environ["GOOGLE_API_KEY"],  # correct parameter
    temperature=0
)

# Send a message
response = llm.invoke("Hi")
print(response.content)

Hi there! How can I help you today?


### Data Ingestion

In [4]:
from langchain_community.document_loaders import TextLoader

In [5]:
loader = TextLoader("E:\\1.3.5.7.9\\2.LLMops\\data\\agenticai.txt", encoding="utf8")
documents = loader.load()

In [6]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'Agentic AI represents a transformative shift in artificial intelligence, moving from simple response-based systems to autonomous, goal-driven agents capable of acting independently over time. Unlike traditional AI, which primarily processes input and produces output, agentic systems can **plan, reason, and execute tasks in multiple steps**, making them far more powerful and adaptable. This concept is rooted in ideas from Reinforcement Learning, where agents learn through interaction and feedback'

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [9]:
text_chunks=text_splitter.split_documents(documents)

In [10]:
text_chunks

[Document(metadata={'source': 'E:\\1.3.5.7.9\\2.LLMops\\data\\agenticai.txt'}, page_content='Agentic AI represents a transformative shift in artificial intelligence, moving from simple response-based systems to autonomous, goal-driven agents capable of acting independently over time. Unlike'),
 Document(metadata={'source': 'E:\\1.3.5.7.9\\2.LLMops\\data\\agenticai.txt'}, page_content='over time. Unlike traditional AI, which primarily processes input and produces output, agentic systems can **plan, reason, and execute tasks in multiple steps**, making them far more powerful and'),
 Document(metadata={'source': 'E:\\1.3.5.7.9\\2.LLMops\\data\\agenticai.txt'}, page_content='more powerful and adaptable. This concept is rooted in ideas from Reinforcement Learning, where agents learn through interaction and feedback, and is now expanded using modern frameworks like'),
 Document(metadata={'source': 'E:\\1.3.5.7.9\\2.LLMops\\data\\agenticai.txt'}, page_content='frameworks like LangChain and Au

In [11]:
! uv pip install faiss-cpu

Using Python 3.12.1 environment at: E:\1.3.5.7.9\2.LLMops\.venv
Checked 1 package in 35ms


In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",   # ✅ FIXED
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [16]:
# chunks = [doc.page_content for doc in text_chunks]

In [17]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    text_chunks,   # directly use documents (better than texts)
    embeddings
)

In [18]:
retriever = vectorstore.as_retriever()

In [22]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Key characteristics of agentic AI include:
--------------------------------------------------
Document 2:
Agentic AI represents a transformative shift in artificial intelligence, moving from simple response-based systems to autonomous, goal-driven agents capable of acting independently over time. Unlike
--------------------------------------------------
Document 3:
over time. Unlike traditional AI, which primarily processes input and produces output, agentic systems can **plan, reason, and execute tasks in multiple steps**, making them far more powerful and
--------------------------------------------------
Document 4:
AI systems requires structured pipelines, robust evaluation methods, and careful integration of tools and models to ensure efficiency and trustworthiness. Overall, agentic AI marks a significant
--------------------------------------------------


In [ ]:
"""# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Use retriever
docs = retriever.invoke("What is the Key Characteristics of Agentic AI?")

# Display results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)"""

Document 1:
Key characteristics of agentic AI include:
--------------------------------------------------
Document 2:
Agentic AI represents a transformative shift in artificial intelligence, moving from simple response-based systems to autonomous, goal-driven agents capable of acting independently over time. Unlike
--------------------------------------------------
Document 3:
over time. Unlike traditional AI, which primarily processes input and produces output, agentic systems can **plan, reason, and execute tasks in multiple steps**, making them far more powerful and
--------------------------------------------------
Document 4:
AI systems requires structured pipelines, robust evaluation methods, and careful integration of tools and models to ensure efficiency and trustworthiness. Overall, agentic AI marks a significant
--------------------------------------------------


In [24]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [26]:
prompt=ChatPromptTemplate.from_template(template)

In [27]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [29]:
from langchain_core.output_parsers import StrOutputParser

In [30]:
output_parser=StrOutputParser()

In [33]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [34]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI signifies a significant evolution in artificial intelligence, transitioning from basic response-based systems to autonomous, goal-driven agents. Unlike traditional AI, which primarily processes input and produces output, agentic systems possess the ability to plan, reason, and execute tasks through multiple steps. This makes them far more powerful and capable of acting independently over time. In practice, an agentic AI system can perform complex workflows with minimal human input. Examples of such workflows include research, report generation, coding assistance, and business automation.'

In [4]:
"""
[project]
name = "2-llmops"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
    "google-generativeai>=0.8.6",
    "ipykernel>=7.2.0",
    "langchain-google-genai>=4.2.1",
]

    """
    
    # old tomal file

'\n[project]\nname = "2-llmops"\nversion = "0.1.0"\ndescription = "Add your description here"\nreadme = "README.md"\nrequires-python = ">=3.12"\ndependencies = [\n    "google-generativeai>=0.8.6",\n    "ipykernel>=7.2.0",\n    "langchain-google-genai>=4.2.1",\n]\n\n    '